<a href="https://colab.research.google.com/github/afngh/transformers/blob/main/transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [43]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from rich.progress import track
import urllib.request
import math
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim

In [11]:
class Embedding(nn.Module):
  def __init__(self, dims: int=64, vocab_size: int=5000, dropout: float=0.3):
    super().__init__()

    self.embed = nn.Embedding(vocab_size, dims)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    return self.dropout(self.embed(x))

In [12]:
class PositionalEmbedding(nn.Module):
  def __init__(self, dims: int=64, vocab_size: int=5000):
    super().__init__()

    pe = torch.zeros(vocab_size, dims)

    position = torch.arange(0, vocab_size, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, dims, 2).float() * (-math.log(10000.0) / dims))

    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)

    self.pe = pe.unsqueeze(0)

  def forward(self, x):
    x = x + self.pe[:, :x.size(1)]
    return x

In [13]:
class Attention(nn.Module):
  def __init__(self, dims: int=64, dropout: float=0.3, head: int=4):
    super().__init__()

    self.dims = dims
    self.head = head
    self.hdim = dims // head

    self.dropout = nn.Dropout(dropout)

    self.q = nn.Linear(dims, dims)
    self.k = nn.Linear(dims, dims)
    self.v = nn.Linear(dims, dims)

    self.projection = nn.Linear(dims, dims)

  def forward(self,x):

    B, S, D = x.size()

    q = self.q(x)
    k = self.k(x)
    v = self.v(x)

    q = q.view(B, S, self.head, self.hdim).transpose(1, 2)
    k = k.view(B, S, self.head, self.hdim).transpose(1, 2)
    v = v.view(B, S, self.head, self.hdim).transpose(1, 2)

    scores = torch.matmul(q,k.transpose(-1,-2) / self.hdim ** 0.5)

    attention_weights = torch.softmax(scores, dim=-1)
    attention_weights = self.dropout(attention_weights)

    x = torch.matmul(attention_weights, v)
    x = x.transpose(1, 2).contiguous().view(B, S, D)
    x = self.projection(x)

    return x

In [14]:
class PostAttention(nn.Module):
  def __init__(self, dims: int=64, dropout: float=0.3):
    super().__init__()

    self.norm1 = nn.LayerNorm(dims)
    self.norm2 = nn.LayerNorm(dims)

    self.fnn = nn.Sequential(
        nn.Linear(dims, dims*4),
        nn.ReLU(),
        nn.Linear(dims*4, dims)
    )

    self.drop1 = nn.Dropout(dropout)
    self.drop2 = nn.Dropout(dropout)

  def forward(self, x, attention):
    x = x + self.drop1(attention)
    x = self.norm1(x)

    f = self.fnn(x)

    x = x + self.drop2(f)
    x = self.norm2(x)

    return x

In [15]:
class Transformer(nn.Module):
  def __init__(self, dims: int=64, vocab_size: int=5000, dropout: float=0.3, head: int=4):
    super().__init__()

    self.output_layer = nn.Linear(dims, vocab_size)

  def forward(self, x):
    return self.output_layer(x)

In [51]:
class Model(nn.Module):
  def __init__(self, EmbeddingModel, PositionalEmbeddingModel, Attention, PostAttention, Transformer):
    super().__init__()

    self.e = EmbeddingModel
    self.pe = PositionalEmbeddingModel
    self.a = Attention
    self.pa = PostAttention
    self.t = Transformer

  def forward(self, x):
    wv = self.e(x)
    pwe = self.pe(wv)
    att = self.a(pwe)
    post = self.pa(pwe, att)

    last_vector = post[:,-1:,:].squeeze(0)
    logits = self.t(last_vector)

    return logits

In [29]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "text.txt")
with open("text.txt") as f:
    text = f.read(5000)

words = text.split()

vocab = sorted(list(set(words)))

wti = {word: i for i, word in enumerate(vocab)}
itw = {i: word for i, word in enumerate(vocab)}

seq = 5

X = []
y = []

for i in range(len(words)-seq):
  X.append([wti[w] for w in words[i:i+seq]])
  y.append([wti[words[i+seq]]])

X = torch.tensor(X)
y = torch.tensor(y)

# x_test = torch.tensor([[21,  15,  10, 450]])
# y_test = torch.tensor([[332]])

In [26]:
e = Embedding(
    vocab_size=len(vocab),
    dims=64,
)

pe = PositionalEmbedding(
    vocab_size=len(vocab),
    dims=64,
)

a = Attention(
    dims=64,
    head=4,
)

pa = PostAttention(
    dims=64,
)

t = Transformer(
    dims=64,
    vocab_size=len(vocab),
)

model = Model(
    EmbeddingModel=e,
    PositionalEmbeddingModel=pe,
    Attention=a,
    PostAttention=pa,
    Transformer=t,
)

torch.Size([1, 482])

In [37]:
dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset,batch_size = 32,shuffle=True)

In [46]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.9)

In [63]:
EPOCHS = 30
NORM = 1.0

for epoch in track(range(30),description="Training Vocab:"):
  model.train()
  el = None
  for X_data,y_true in dataloader:
    y_pred = model(X_data)
    # print(y_pred.squeeze(1).shape)
    # print(y_true.shape)
    loss = loss_fn(y_pred.squeeze(1),y_true.squeeze(1))
    optimizer.zero_grad()
    el = loss.item()
    loss.backward()
    optimizer.step()
  print(f"Epoch: {epoch} && Loss: {el}")
  scheduler.step()

Output()

torch.Size([32, 482])

torch.Size([32, 1])

torch.Size([32, 482])

torch.Size([32, 1])

...

torch.Size([32, 482])

torch.Size([32, 1])

torch.Size([14, 482])

torch.Size([14, 1])

Epoch: 29 && Loss: 1.8765183687210083